# 02 · Order-flow imbalance (Cont, Kukanov & Stoikov 2014)

Per-event term computed in C++ during replay:

$e_n = \mathbb{1}[P^b_n \ge P^b_{n-1}]\,q^b_n - \mathbb{1}[P^b_n \le P^b_{n-1}]\,q^b_{n-1} - \mathbb{1}[P^a_n \le P^a_{n-1}]\,q^a_n + \mathbb{1}[P^a_n \ge P^a_{n-1}]\,q^a_{n-1}$

CKS show that over a bucket, $\Delta P_k \approx \beta\,\mathrm{OFI}_k$ with $\beta \propto 1/\text{depth}$.
Two checks: the contemporaneous regression (does OFI explain price changes?) and an
event-time predictive check (does past OFI correlate with *future* mid changes?).

In [ ]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "python"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
import lob
from lob import data as D
from lob.reconstruct import replay_day
RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
print("lob", lob.__version__, "| tickers:", [d.ticker for d in D.find_days()])

In [ ]:
from lob.research import bucket_sweep, ofi_regression, predictive_table, plot_ofi
pd.concat([bucket_sweep(t) for t in ["AMZN","AAPL","GOOG","INTC","MSFT"]]).round(5)

`slope_x_depth` is roughly constant *within* the small-tick (AMZN/AAPL/GOOG) and
large-tick (INTC/MSFT) groups, which is the CKS depth-scaling prediction. R² rises
with the bucket size, as in the paper.

In [ ]:
for t in ["AMZN","INTC"]:
    plot_ofi(t, 10.0, RESULTS/f"ofi_{t.lower()}.png"); plt.show()

In [ ]:
pred = pd.concat([predictive_table(t) for t in ["AMZN","AAPL","INTC"]])
pred.pivot_table(index=["ticker","signal"], columns="horizon_events", values="corr").round(4)

Predictive correlations are positive but small (a few percent). Contemporaneous
explanatory power is much larger than predictive power, which is the first hint that a
naive OFI-threshold strategy will struggle net of spread and adverse selection.